# 16S Analyses of the Longitudinal Acne Study
## V1-V3 and V4 primer sets comparison

Date created: 12/28/2024

Notebook authors: Yang Chen

Data analysis by: Yang Chen, Tyler Myers, Britta De Pessemier

This notebook plots the following:

- Plot showing abundance of Cutibacterium in each sample with each primer pair (i.e. the axes are V13 vs V4, each point is the amount of Cutibacterium in one sample by each of the primer pairs)
- Venn diagram illustrating the overlap of genera-level taxa detected by both primer sets, alongside those unique to V1-V3 or V4

In [99]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import gemelli
from gemelli.preprocessing import rclr_transformation
from matplotlib_venn import venn2, venn2_circles
from Bio import SeqIO
from matplotlib.patches import Circle
from scipy.stats import pearsonr
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import pearsonr, norm
from sklearn.decomposition import PCA
from biom import Table


In [100]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    return df


In [101]:
# Load tables
V1V3_biom = biom_to_df('../Data/16S/Tables/16S_V1-V3_Genus_collapsed.biom')
V4_biom = biom_to_df('../Data/16S/Tables/16S_V4_Genus_collapsed.biom')
V1V3_biom.index = V1V3_biom.index.str.replace('^ g__', '', regex=True)
V4_biom.index = V4_biom.index.str.replace('^ g__', '', regex=True)
V1V3_biom

,LAMI.RD001.D0.C1,LAMI.RD001.D0.C2,LAMI.RD001.D14.C1,LAMI.RD001.D14.C2,LAMI.RD001.D28.C1,LAMI.RD002.D0.C2,LAMI.RD003.D14.C1,LAMI.RD002.D14.C1,LAMI.RD003.D28.C1,LAMI.RD001.D28.C2,...,LAMI.RD319.D28.C3,LAMI.RD319.D11.C1,LAMI.RD319.D21.C2,LAMI.RD319.D7.C3,LAMI.RD318.D28.C3,LAMI.RD319.D14.C1,LAMI.RD319.D21.C3,LAMI.RD319.D14.C2,LAMI.RD319.D9.C1,LAMI.RD319.D9.C2
Cutibacterium,0.626691,0.479703,0.592730,0.467233,0.628018,0.769435,0.874503,0.912709,0.628018,0.607058,...,0.447334,0.346246,0.692757,0.260281,0.853807,0.504112,0.266118,0.614221,0.326347,0.613956
uncultured,0.000000,0.000531,0.000531,0.000000,0.002919,0.000000,0.013797,0.001327,0.033696,0.000000,...,0.537543,0.642080,0.297957,0.732024,0.142743,0.489785,0.727249,0.374105,0.660918,0.374370
Staphylococcus,0.024940,0.041921,0.022022,0.036349,0.031308,0.098965,0.025206,0.027859,0.027328,0.054126,...,0.002653,0.000000,0.002653,0.005572,0.001327,0.002123,0.002919,0.002123,0.003449,0.002123
Streptococcus,0.077474,0.099231,0.063677,0.118334,0.077739,0.042186,0.003715,0.011409,0.144070,0.044839,...,0.000000,0.002123,0.000000,0.000000,0.000000,0.000000,0.000265,0.000000,0.000265,0.000000
Corynebacterium,0.046431,0.093393,0.130539,0.120987,0.096843,0.010348,0.036614,0.013001,0.008225,0.105864,...,0.002653,0.003715,0.001857,0.000531,0.000000,0.001592,0.001061,0.002388,0.000796,0.003715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Methylotenera,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Fusicatenibacter,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Ruminococcus,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Lachnospira,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [102]:
# Convert indices to sets for set operations
indices_V1V3 = set(V1V3_biom.index)
indices_V4 = set(V4_biom.index)

# Taxa unique to V1V3
unique_to_V1V3 = indices_V1V3 - indices_V4

# Taxa unique to V4
unique_to_V4 = indices_V4 - indices_V1V3

# Taxa shared by both
shared_taxa = indices_V1V3 & indices_V4

# Step 2: Filter shared and unique taxa based on 10% sample prevalence
def filter_by_prevalence(df, taxa, prevalence_threshold=0.1):
    # Subset the DataFrame to include only the specified taxa
    subset_df = df.loc[list(taxa)]  # Convert set to list for indexing
    # Calculate prevalence: proportion of samples where the taxon is present
    prevalence = (subset_df > 0).sum(axis=1) / subset_df.shape[1]
    # Filter taxa based on the prevalence threshold
    return set(prevalence[prevalence >= prevalence_threshold].index)

# Apply prevalence filtering
filtered_shared = filter_by_prevalence(V1V3_biom, shared_taxa, prevalence_threshold=0.1)
filtered_unique_to_V1V3 = filter_by_prevalence(V1V3_biom, unique_to_V1V3, prevalence_threshold=0.1)
filtered_unique_to_V4 = filter_by_prevalence(V4_biom, unique_to_V4, prevalence_threshold=0.1)

# Step 3: Create the Venn diagram with enhancements
plt.clf()
plt.figure(figsize=(8, 8))  # Increased figure size for better clarity

# Create the Venn diagram with customized circle outlines
venn = venn2(
    [filtered_unique_to_V1V3 | filtered_shared, filtered_unique_to_V4 | filtered_shared],
    ('', ''),
    set_colors=('lightblue', 'lightgreen'),  # Fill colors for the circles
    alpha=0.4  # Transparency for fill colors
)

# Customize the circle outlines with darker colors
for circle, color in zip(['10', '01'], ['blue', 'green']):
    venn.get_patch_by_id(circle).set_edgecolor(color)  # Darker outline color
    venn.get_patch_by_id(circle).set_linewidth(2)  # Thickness of the outline

# Customizing colors for the Venn diagram (switching green and purple)
venn.get_patch_by_id('10').set_color('#87CEEB')  # Light blue for V1-V3
venn.get_patch_by_id('01').set_color('#DDA0DD')  # Light purple for V4
venn.get_patch_by_id('11').set_color('#98FB98')  # Light green for shared

# Customizing text labels
for id in ['10', '01', '11']:
    if venn.get_label_by_id(id):
        venn.get_label_by_id(id).set_fontsize(12)  # Larger font size
        venn.get_label_by_id(id).set_color('black')  # Black text for better readability

# Add a title with larger font size and weight
plt.title('Bacterial Genera Resolved by 16S V1-V3 vs V4',
          fontsize=18)

# Add a legend for the groups
plt.legend(
    handles=[
        plt.Line2D([0], [0], marker='o', color='#87CEEB', lw=0, label='V1-V3 Unique'),
        plt.Line2D([0], [0], marker='o', color='#98FB98', lw=0, label='Shared'),
        plt.Line2D([0], [0], marker='o', color='#DDA0DD', lw=0, label='V4 Unique'),
    ],
    loc='lower center', bbox_to_anchor=(0.5, -0.075), ncol=3, fontsize=12
)

# Save the figure with a higher resolution
plt.savefig('../Figures/16S_Figures/primer_comparison/primer_venn_diagram.png', dpi=600, bbox_inches='tight')

# Show the figure (optional)
plt.show()

# Print the results
print("Filtered Unique to V1V3 (>=10% prevalence):")
print(filtered_unique_to_V1V3)

print("\nFiltered Unique to V4 (>=10% prevalence):")
print(filtered_unique_to_V4)

print("\nFiltered Shared Taxa (>=10% prevalence):")
print(filtered_shared)


Filtered Unique to V1V3 (>=10% prevalence):
{'Janibacter', 'Mycobacterium', 'Reyranella'}

Filtered Unique to V4 (>=10% prevalence):
{'Fenollaria', 'Dolosigranulum', 'Atopobium', 'Marinomonas', 'Chryseobacterium', 'Empedobacter', 'Capnocytophaga', 'Frederiksenia', 'Pantoea', 'Leptotrichia', 'Stenotrophomonas', 'Turicella', 'Enhydrobacter', 'Fusobacterium', 'Luteimonas', 'Bradyrhizobium', 'Lactococcus', 'Massilia', 'Finegoldia', 'Abiotrophia', 'Peptostreptococcus', 'Brachybacterium', 'Geobacillus', 'Psychrobacter', 'Jeotgalicoccus', 'Gardnerella', 'Vibrio', 'Alloiococcus', 'Moraxella', 'Leuconostoc', 'Peptoniphilus', 'Aerococcus', 'Aeromonas', 'Aggregatibacter', 'Blautia', 'Bifidobacterium'}

Filtered Shared Taxa (>=10% prevalence):
{'Corynebacterium', 'uncultured', 'Gemella', 'Lysobacter', 'Veillonella', 'Phenylobacterium', 'Sphingopyxis', 'Limnobacter', 'Anaerococcus', 'Pseudomonas', 'Brevundimonas', 'Nocardioides', 'Brevibacterium', 'Micrococcus', 'Thermus', 'Rothia', 'Staphylococcus

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_38664/4290181383.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [103]:
# # Read and transform V1-V3 and V4 Genus level absolute abundance table
# V1V3_biom = biom_to_df('../Data/16S/Tables/16S_V1-V3_Genus_collapsed.biom')
# V4_biom = biom_to_df('../Data/16S/Tables/16S_V4_Genus_collapsed.biom')

### Comparison of 16S V1-V3, 16S V4, and shotgun of Cutibacterium per-sample ranked relative abundances with intersectional samples

In [104]:
def get_taxon_ranked_df(taxon_name, v1v3_df, v4_df, shotgun_df):
    # Format the taxon name for each source
    v1v3_taxon = f'{taxon_name}'
    v4_taxon = f'{taxon_name}'
    shotgun_taxon = f'{taxon_name}'
    
    if v1v3_taxon not in v1v3_df.index or v4_taxon not in v4_df.index or not shotgun_taxon:
        raise ValueError(f"Taxon '{taxon_name}' not found in one or more datasets.")
    
    # Extract and transpose
    v1v3_data = v1v3_df.loc[[v1v3_taxon]].T
    v1v3_data.columns = [f'{taxon_name} V1-V3']
    v1v3_data['V1-V3'] = v1v3_data.iloc[:, 0].rank(method='min').astype(int)

    v4_data = v4_df.loc[[v4_taxon]].T
    v4_data.columns = [f'{taxon_name} V4']
    v4_data['V4'] = v4_data.iloc[:, 0].rank(method='min').astype(int)

    shot_data = shotgun_df[[shotgun_taxon]].rename(columns={shotgun_taxon: f'{taxon_name} Shotgun'})
    shot_data['Shotgun'] = shot_data.iloc[:, 0].rank(method='min').astype(int)

    # Standardize index names across datasets
    shot_data.index = shot_data.index.str.replace('_', '.', regex=False)
    v1v3_data.index = v1v3_data.index.astype(str)
    v4_data.index = v4_data.index.astype(str)
    
    # Align sample names
    shared_samples = v1v3_data.index.intersection(v4_data.index).intersection(shot_data.index)
    
    # Merge all
    merged = pd.concat([
        v1v3_data.loc[shared_samples]['V1-V3'],
        v4_data.loc[shared_samples]['V4'],
        shot_data.loc[shared_samples]['Shotgun']
    ], axis=1)
    
    return merged


In [105]:
# Load tables
shotgun_tbl = pd.read_csv('../Data/metaG/VEBA_analysis/output_files/X_mags.with_taxonomy.with_slcs_name.csv')
shotgun_tbl_subset = shotgun_tbl.iloc[:, 4:]
shotgun_tbl_subset = shotgun_tbl_subset.set_index('name')
shotgun_tbl_subset.index.name = None

# shotgun_tbl = biom_to_df('../Data/metaG/Tables/metaG_rs210_micov-filt_genus.biom')
shotgun_tbl



,SLC,sample_mag,organism_type,taxonomy,name,LAMI_RD308_D9_C3_S18_L005,LAMI_RD306_D28_C3_S12_L005,LAMI_RD310_D16_C2_S22_L005,LAMI_RD304_D14_C3_S4_L005,LAMI_RD304_D11_C1_S3_L005,...,LAMI_RD304_D28_C3_S5_L005,LAMI_RD308_D0_C2_S13_L005,LAMI_RD310_D0_C2_S19_L005,LAMI_RD306_D11_C1_S8_L005,LAMI_RD310_D0_C3_S20_L005,LAMI_RD310_D14_C3_S21_L005,LAMI_RD306_D14_C1_S9_L005,LAMI_RD310_D21_C2_S23_L005,LAMI_RD308_D14_C3_S16_L005,LAMI_RD304_D0_C1_S1_L005
0,PSLC-1,LAMI_RD306_D0_C3_S7_L005__MAXBIN2-40__P.1__bin...,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,Cutibacterium acnes,777667,1513946,19467,127037,5905,...,362236,1811,196880,825942,372845,441798,83571,196812,782381,111499
1,PSLC-2,LAMI_RD306_D0_C3_S7_L005__METABAT2__P.1__bin.2,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,Cutibacterium granulosum,14720,171159,1374,21838,896,...,60647,27,20516,65240,26311,13760,5157,28370,7988,23177
2,PSLC-7,LAMI_RD306_D11_C1_S8_L005__CONCOCT__P.1__14_sub,prokaryotic,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,Other (combined),5,74,0,35,1,...,57,0,1,102859,5,1,22,13,1,35
3,PSLC-11,LAMI_RD306_D11_C1_S8_L005__CONCOCT__P.1__26,prokaryotic,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,Other (combined),31,69,0,3,0,...,4,0,1,764138,0,0,25,0,3,1
4,PSLC-3,LAMI_RD306_D11_C1_S8_L005__CONCOCT__P.1__27,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,Lawsonella cleavelandensis,1217,72530,123,7890,258,...,4476,4,661,94133,819,446,11267,970,635,2289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,PSLC-1,LAMI_RD310_D16_C2_S22_L005__METABAT2__P.1__bin.1,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,Cutibacterium acnes,325344,98077,34139,416073,20568,...,1305974,694,551756,70551,674986,371153,4985,705039,212634,404418
73,PSLC-1,LAMI_RD310_D21_C2_S23_L005__CONCOCT__P.1__37,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,Cutibacterium acnes,389589,171086,49066,459679,22935,...,1430689,832,848529,112684,879014,549818,8742,1076846,268975,448362
74,VSLC-6,LAMI_RD310_D7_C3_S24_L005__GENOMAD__Virus.1,viral,Viruses;Monodnaviria;Shotokuvirae;Cossaviricot...,Papillomaviridae,0,38,0,0,0,...,0,0,4,10,6305,0,0,0,0,0
75,PSLC-1,LAMI_RD310_D7_C3_S24_L005__METABAT2__P.1__bin.1,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,Cutibacterium acnes,328321,91234,35262,438165,21590,...,1370474,659,574052,67200,719147,381184,4591,726124,209969,424201


In [106]:
shotgun_tbl_subset_collapsed = shotgun_tbl_subset.groupby(shotgun_tbl_subset.index).sum()
shotgun_tbl_subset_collapsed = shotgun_tbl_subset_collapsed.T
shotgun_tbl_subset_collapsed

# Save as the VEBA MAGs collapsed Genus-level
biom_table = Table(
    data=shotgun_tbl_subset_collapsed.values,
    observation_ids=shotgun_tbl_subset_collapsed.index.tolist(),
    sample_ids=shotgun_tbl_subset_collapsed.columns.tolist()
)

# Save to file
output_fp = "../Data/metaG/Tables/VEBA_MAGs_collapsed_genus_absolute.biom"
with open(output_fp, "w") as f:
    biom_table.to_json("Generated from pandas DataFrame", f)

In [107]:
relative_abundance_df = shotgun_tbl_subset_collapsed.div(shotgun_tbl_subset_collapsed.sum(axis=0), axis=1)
relative_abundance_df = relative_abundance_df

In [108]:
# Fix sample names
relative_abundance_df.index = (
    relative_abundance_df.index
    .str.replace('_', '.', regex=False)
    .str.split('.')
    .str[:4]
    .str.join('.')
)
relative_abundance_df

,Caudoviricetes,Corynebacterium spp.,Cutibacterium acnes,Cutibacterium granulosum,Lawsonella cleavelandensis,Malassezia spp.,Micrococcus spp.,Neisseria spp.,Other (combined),Papillomaviridae,Streptococcus mitis
LAMI.RD308.D9.C3,0.001234,0.031088,0.073499,0.016521,0.005557,0.011224,0.095239,0.033008,0.024552,0.000059,0.081239
LAMI.RD306.D28.C3,0.187919,0.177730,0.067423,0.317516,0.100605,0.134472,0.245995,0.116269,0.046554,0.497668,0.039488
LAMI.RD310.D16.C2,0.002335,0.002159,0.002994,0.001109,0.000366,0.004741,0.000952,0.001557,0.000535,0.000000,0.000439
LAMI.RD304.D14.C3,0.002142,0.020056,0.032211,0.020084,0.012521,0.016738,0.016177,0.027895,0.012495,0.000088,0.130071
LAMI.RD304.D11.C1,0.000127,0.001040,0.001629,0.000708,0.000454,0.001363,0.001459,0.001588,0.000568,0.000010,0.001667
LAMI.RD306.D14.C3,0.199648,0.109615,0.068786,0.149568,0.467024,0.167152,0.069023,0.182014,0.034594,0.146779,0.051562
LAMI.RD308.D0.C3,0.000107,0.001630,0.005007,0.000959,0.000390,0.000083,0.002186,0.000680,0.000279,0.000000,0.000487
LAMI.RD306.D23.C1,0.038412,0.130397,0.077452,0.020772,0.030145,0.011146,0.012941,0.278063,0.005018,0.011536,0.004336
LAMI.RD308.D23.C2,0.044754,0.031542,0.036889,0.012899,0.082071,0.006559,0.035832,0.050277,0.190533,0.000013,0.574321
LAMI.RD310.D7.C3,0.001034,0.002069,0.003265,0.000983,0.000266,0.000583,0.000159,0.000437,0.000410,0.007610,0.000021


In [109]:
# Make a copy to avoid modifying the original
collapsed_df = relative_abundance_df.copy()

# Collapse Cutibacterium acnes and granulosum into one column
collapsed_df['Cutibacterium'] = (
    collapsed_df.get('Cutibacterium acnes', 0) + 
    collapsed_df.get('Cutibacterium granulosum', 0)
)

# Rename other specific columns
collapsed_df = collapsed_df.rename(columns={
    'Corynebacterium spp.': 'Corynebacterium',
    'Lawsonella cleavelandensis': 'Lawsonella',
    'Micrococcus spp.': 'Micrococcus',
    'Streptococcus mitis': 'Streptococcus'
})

# Drop the old individual columns that were collapsed or renamed
collapsed_df = collapsed_df.drop(columns=[
    'Cutibacterium acnes',
    'Cutibacterium granulosum'
])

# (Optional) View the updated column names
print(collapsed_df.columns.tolist())


['Caudoviricetes', 'Corynebacterium', 'Lawsonella', 'Malassezia spp.', 'Micrococcus', 'Neisseria spp.', 'Other (combined)', 'Papillomaviridae', 'Streptococcus', 'Cutibacterium']


In [110]:
V1V3_biom.columns

Index(['LAMI.RD001.D0.C1', 'LAMI.RD001.D0.C2', 'LAMI.RD001.D14.C1',
       'LAMI.RD001.D14.C2', 'LAMI.RD001.D28.C1', 'LAMI.RD002.D0.C2',
       'LAMI.RD003.D14.C1', 'LAMI.RD002.D14.C1', 'LAMI.RD003.D28.C1',
       'LAMI.RD001.D28.C2',
       ...
       'LAMI.RD319.D28.C3', 'LAMI.RD319.D11.C1', 'LAMI.RD319.D21.C2',
       'LAMI.RD319.D7.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD319.D14.C1',
       'LAMI.RD319.D21.C3', 'LAMI.RD319.D14.C2', 'LAMI.RD319.D9.C1',
       'LAMI.RD319.D9.C2'],
      dtype='object', length=236)

In [111]:
collapsed_df.columns

Index(['Caudoviricetes', 'Corynebacterium', 'Lawsonella', 'Malassezia spp.',
       'Micrococcus', 'Neisseria spp.', 'Other (combined)', 'Papillomaviridae',
       'Streptococcus', 'Cutibacterium'],
      dtype='object')

In [112]:
# Get shared sample IDs across all three datasets
shared_samples = (
    V1V3_biom.columns
    .intersection(V4_biom.columns)
    .intersection(collapsed_df.index)
)

# Confirm sample count
print(f"Number of matched samples: {len(shared_samples)}")

# Subset each dataset
V1V3_biom = V1V3_biom[shared_samples]
V4_biom = V4_biom[shared_samples]
collapsed_df = collapsed_df.loc[shared_samples]

Number of matched samples: 22


In [113]:
taxa_info = {
    "Lawsonella": {"color": "#4169e1"},
    "Cutibacterium": {"color": "#ff7f00"},
    "Streptococcus": {"color": "#c8a165"},
    "Micrococcus": {"color": "#f47fb8"},
    "Corynebacterium": {"color": "#fff44f"}
}

In [114]:
fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection='3d')

legend_items = []

for taxon, style in taxa_info.items():
    try:
        # Get sample-wise ranked data for this taxon
        df = get_taxon_ranked_df(
            taxon_name=taxon,
            v1v3_df=V1V3_biom,
            v4_df=V4_biom,
            shotgun_df=collapsed_df
        )

        x, y, z = df['V1-V3'].values, df['V4'].values, df['Shotgun'].values

        # PCA-style best-fit line in 3D
        points = np.stack([x, y, z], axis=1)
        mean_point = points.mean(axis=0)
        centered_points = points - mean_point
        pca_line = PCA(n_components=1)
        pca_line.fit(centered_points)
        direction = pca_line.components_[0]

        # Create and plot PCA line
        t = np.linspace(-20, 20, 100)
        line = mean_point + np.outer(t, direction)
        ax.plot(line[:, 0], line[:, 1], line[:, 2],
                color=style['color'], linewidth=2, alpha=0.8)

        # Compute multivariate R²
        pca_full = PCA(n_components=3)
        pca_full.fit(centered_points)
        r_squared = pca_full.explained_variance_ratio_[0]

        # Plot scatter points and collect legend item
        scatter = ax.scatter(x, y, z, color=style['color'],
                             edgecolors='k', s=50, alpha=0.7)
        legend_label = f"{taxon} (R² = {r_squared:.2f})"
        legend_items.append((scatter, legend_label, r_squared))

    except Exception as e:
        print(f"Skipped {taxon}: {e}")

# Sort legend items by R² descending
legend_items_sorted = sorted(legend_items, key=lambda x: x[2], reverse=True)
handles, labels = zip(*[(item[0], item[1]) for item in legend_items_sorted])

# Plot styling
ax.set_xlabel('16S V1–V3 Rank', fontsize=16)
ax.set_ylabel('16S V4 Rank', fontsize=16)
ax.set_zlabel('Shotgun Rank', fontsize=16)
ax.set_title('Cross-platform Correlation per Taxon', fontsize=20)

# Set tick label font sizes
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.tick_params(axis='z', labelsize=12)

# Remove negative axis values
ax.set_xlim(0, ax.get_xlim()[1])
ax.set_ylim(0, ax.get_ylim()[1])
ax.set_zlim(0, ax.get_zlim()[1])

# Add legend
ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(-0.05, 0.9), fontsize=12)

# Final output
plt.tight_layout()
plt.savefig('../Figures/multi-omics_Figures/3D-plot_16SV1V3_16SV4_shotgun_top-taxa-corr-R2.png', dpi=600)
plt.show()

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_38664/3055842496.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
